In [1]:
!uv pip install tsfresh

Using Python 3.13.2 environment at: /home/petelin/TSCGlue/.venv
Audited 1 package in 11ms


In [2]:
from aeon.transformations.collection.feature_based import TSFresh
from tscglue.data_loader import load_fold, DATA_DIR
import os
import numpy as np

In [3]:
datasets = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print(f"{len(datasets)} datasets")
datasets[:10]

112 datasets


['ACSF1',
 'Adiac',
 'ArrowHead',
 'BME',
 'Beef',
 'BeetleFly',
 'BirdChicken',
 'CBF',
 'Car',
 'Chinatown']

In [4]:
ds = datasets[0]
X_train, y_train, X_test, y_test = load_fold(ds, fold=0)
print(f"Dataset: {ds}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

Dataset: ACSF1
X_train shape: (100, 1, 1460)
X_test shape:  (100, 1, 1460)


In [5]:
# Compare minimal, efficient, and comprehensive feature sets
#for fc_params in ["minimal", "efficient", "comprehensive"]:
#    tf = TSFresh(default_fc_parameters=fc_params)
#    X_tf = tf.fit_transform(X_train)
#    print(f"{fc_params:15s} -> {X_tf.shape[1]:>6} features")

In [6]:
# Extract efficient features and inspect
tf = TSFresh(default_fc_parameters="efficient", n_jobs=-1)
X_tf_train = tf.fit_transform(X_train)
X_tf_test = tf.transform(X_test)
print(f"Train features shape: {X_tf_train.shape}")
print(f"Test features shape:  {X_tf_test.shape}")
print(f"Any NaNs in train: {np.isnan(X_tf_train).any()}")
print(f"Any NaNs in test:  {np.isnan(X_tf_test).any()}")

Train features shape: (100, 777)
Test features shape:  (100, 777)
Any NaNs in train: False
Any NaNs in test:  False


In [7]:
# Quick classification with Ridge on TSFresh features
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

clf = make_pipeline(StandardScaler(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
clf.fit(X_tf_train, y_train)
y_pred = clf.predict(X_tf_test)
print(f"Dataset: {ds}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Dataset: ACSF1
Accuracy: 0.7700


In [8]:
# Feature counts across datasets
from time import perf_counter

for ds in datasets[::10]:
    X_train, y_train, _, _ = load_fold(ds, fold=0)
    tf = TSFresh(default_fc_parameters="efficient")
    t0 = perf_counter()
    n_feats = tf.fit_transform(X_train).shape[1]
    dur = perf_counter() - t0
    print(f"{ds:40s}  X={str(X_train.shape):20s}  tsfresh_efficient={n_feats:>6}  {dur:.2f}s")

ACSF1                                     X=(100, 1, 1460)        tsfresh_efficient=   777  28.61s
ChlorineConcentration                     X=(467, 1, 166)         tsfresh_efficient=   777  64.57s
DistalPhalanxOutlineCorrect               X=(600, 1, 80)          tsfresh_efficient=   777  73.45s
FaceAll                                   X=(560, 1, 131)         tsfresh_efficient=   777  77.33s
GunPointAgeSpan                           X=(135, 1, 150)         tsfresh_efficient=   777  19.69s
InsectEPGSmallTrain                       X=(17, 1, 601)          tsfresh_efficient=   777  3.35s
MiddlePhalanxOutlineCorrect               X=(600, 1, 80)          tsfresh_efficient=   777  76.75s
Phoneme                                   X=(214, 1, 1024)        tsfresh_efficient=   777  43.11s
Rock                                      X=(20, 1, 2844)         tsfresh_efficient=   777  9.12s
SonyAIBORobotSurface2                     X=(27, 1, 65)           tsfresh_efficient=   777  3.17s
TwoPatterns  